In [4]:
import torch
from torch import Tensor
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import tiktoken

class WMT(Dataset):
    def __init__(self, data_split, num_rows):
        super().__init__()
        dataset = load_dataset("wmt/wmt14", "fr-en", split=f"{data_split}[:{num_rows}]")
        self.df = dataset.data.to_pandas()
        self.rows: list[tuple[Tensor, Tensor] | None] = [None for _ in range(num_rows)]
        self.encoder = tiktoken.get_encoding("cl100k_base")

    def __len__(self):
        return self.df.shape[0]

    def vocab_size(self):
        return self.encoder.n_vocab

    def __getitem__(self, idx):
        if self.rows[idx] is None:
            c = self.df.columns[0]
            en, fr = self.df[c].str["en"][idx], self.df[c].str["fr"][idx]
            en_tokens, fr_tokens = self.encoder.encode(en), self.encoder.encode(fr)
            self.rows[idx] = (torch.tensor(en_tokens), torch.tensor(fr_tokens))
        return self.rows[idx][0], self.rows[idx][1]

/home/aabiji/dev/ml/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
class TokenEmbedding(nn.Module):
    def __init__(self, num_tokens, d_model):
        super().__init__()
        # Positional encoding matrix
        div_term = 10000 ** (-torch.arange(0, d_model) * (2 / d_model))
        pos = torch.arange(0, num_tokens).view(-1, 1).expand(num_tokens, d_model) * div_term

        self.positional = torch.zeros(num_tokens, d_model)
        self.positional[:, 0::2] = torch.sin(pos[:, 0::2])
        self.positional[:, 1::2] = torch.cos(pos[:, 1::2])

        # Embedding matrix
        self.weights = torch.rand(num_tokens, d_model)

    def forward(self, x):
        # Map each token (scalar) to the corresponding embedding vector
        # x.shape == (batch_size, num_tokens), self.weights.shape == (num_tokens, d_model),
        # result.shape == (batch_size, num_tokens, d_model)
        return self.weights[x] + self.positional


class MultiHeadAttention(nn.Module):
    def __init__(self, B, d_model, d_k, d_v, h):
        super().__init__()
        self.W_q = torch.rand(B, h, d_model, d_k)
        self.W_k = torch.rand(B, h, d_model, d_k)
        self.W_v = torch.rand(B, h, d_model, d_v)
        self.W_o = torch.rand(B, h * d_v, d_model)

    def forward(self, Q, K, V):
        # Project Q, K, V into smaller subspaces using each h projection matrices
        Q_proj = torch.einsum("bij,bhjk->bhik", Q, self.W_q)
        K_proj = torch.einsum("bij,bhjk->bhik", K, self.W_k)
        V_proj = torch.einsum("bij,bhjk->bhik", V, self.W_v)

        # Compute attention for each attention head:
        # Q @ K.T for each attention head in each batch to get similarity matrices
        a = torch.einsum("bhij,bhkj->bhik", Q_proj, K_proj)
        # Scale by 1 / sqrt(d_k) to prevent the dot product from exploding
        b = torch.exp(a / K.shape[-1])
        # Sum over rows of the similarity matrices. Each row in 'c' corresponds to
        # an attention head, and each column corresponds to a row sum
        c = torch.sum(b, dim=3)
        # Divide each row in the similarity matrices by their sum
        d = torch.einsum("bhij,bhi->bhij", b, 1 / c)
        # softmax(Q @ K.T) @ V for each attention head in each batch to get the scaled values
        S = torch.einsum("bhij,bhjk->bhik", d, V_proj)

        # Concatenate the scaled values for each attention head together and
        # project the resulting tensor into the original vector space
        B, h, n_v, d_v = S.shape
        concat = torch.permute(S, (0, 2, 1, 3)).reshape(B, n_v, h * d_v)
        return torch.einsum("bij,bjk->bik", concat, self.W_o)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, B, d_model, h, num_tokens):
        super().__init__()
        self.attn = MultiHeadAttention(B, d_model, d_model / h, d_model / h, h),
        self.norm1 = nn.LayerNorm([B, num_tokens, d_model])
        self.l1 = nn.Linear(d_model, d_model * 4)
        self.l2 = nn.Linear(d_model * 4, d_model)
        self.norm2 = nn.LayerNorm([B, num_tokens, d_model])

    def forward(self, x):
        # Self-attention, residual connection and LayerNorm
        attn_out = self.attn(x, x, x)
        attn_out = self.norm1(attn_out + x)
        # Feed forward network, residual connection and Layer Norm
        ff_out = self.l1(attn_out)
        ff_out = self.l2(ff_out)
        return self.norm2(ff_out + attn_out)


class Decoder(nn.Module):
    def __init__(self, B, d_model, h, num_tokens):
        super().__init__()
        self.attn1 = MultiHeadAttention(B, d_model, d_model / h, d_model / h, h)
        self.norm1 = nn.LayerNorm([B, num_tokens, d_model])
        self.attn2 = MultiHeadAttention(B, d_model, d_model / h, d_model / h, h)
        self.norm2 = nn.LayerNorm([B, num_tokens, d_model])
        self.l1 = nn.Linear(d_model, d_model * 4)
        self.l2 = nn.Linear(d_model * 4, d_model)
        self.norm3 = nn.LayerNorm([B, num_tokens, d_model])

    def forward(self, x):
        # Masked self-attention, residual connection and LayerNorm
        attn_out1 = self.attn1(x, x, x)
        attn_out1 = self.norm1(attn_out1 + x)
        # Self-attention, residual connection and LayerNorm
        attn_out2 = self.attn2(attn_out1, attn_out1, attn_out1)
        attn_out2 = self.norm2(attn_out2 + attn_out1)
        # Feed forward network, residual connection and LayerNorm
        ff_out = self.l1(attn_out2)
        ff_out = self.l1(ff_out)
        return self.norm3(ff_out + attn_out2)

In [32]:
class Transformer(nn.Module):
    def __init__(self, B, num_tokens, vocab_size):
        super().__init__()
        d_model, h, N = 512, 8, 6
        self.tokem = TokenEmbedding(num_tokens, d_model)
        self.encoder_layers = nn.ModuleList([Encoder(B, d_model, h, num_tokens)] * N)
        self.decoder_layers = nn.ModuleList([Decoder(B, d_model, h, num_tokens)] * N)
        self.token_out = nn.Linear(d_model, vocab_size)

    def forward(self):
        pass

In [ ]:
# TODO: debug and train!!